### Final Project: Detection of Rare Fetal Congenital Heart Diseases (CHD) Using CARDIUM Ultrasound Images and FetalCLIP: A Deep-Learning Model for Rural Coloradans
#### CSCI 5922
#### Spring 2026
#### Meghna Nag and Vanessa Thorsten

1. Mount Drive and locate CARDIUM data

In [ ]:
# ============================================================
# Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
from pathlib import Path

print("Scanning Drive...\n")

all_files = list(Path("/content/gdrive/MyDrive").rglob("*"))

print("Total files found:", len(all_files))
print("\nRelevant CARDIUM files:\n")

for f in all_files:
    name = f.name.lower()
    if (
        "cardium" in name
        or name.endswith(".json")
        or name.endswith(".tar.gz")
        or name.endswith(".zip")
    ):
        print(f)

Scanning Drive...

Total files found: 7

Relevant CARDIUM files:

/content/gdrive/MyDrive/CARDIUM
/content/gdrive/MyDrive/CARDIUM_Code
/content/gdrive/MyDrive/Colab Notebooks/CARDIUM
/content/gdrive/MyDrive/Colab Notebooks/CARDIUM/CARDIUM_tabular_embeddings


In [ ]:
from pathlib import Path

print("CARDIUM folder contents:")
for f in Path("/content/gdrive/MyDrive/CARDIUM").iterdir():
    print(repr(f.name))

CARDIUM folder contents:
'CARDIUM dataset-20260318T201738Z-3-001.zip'
'CARDIUM_dataset-002.tar.gz'
'cardium_images-003.tar.gz'
'README_CARDIUM.md'
'cardium_clinical_data_wnm_translated_final_cleaned.json'
'cardium_clinical_data_woe_wnm_standarized_f_normalized.json'
'FetalCLIP_weights.pt'
'FetalCLIP_config.json'
'CARDIUM'
'CARDIUM_tabular_embeddings'
'CARDIUM_image_embeddings'
'outputs_multimodal'


In [ ]:
# ============================================================
# Correct CARDIUM paths
# ============================================================

from pathlib import Path
import zipfile
import tarfile
import json
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

DRIVE_ROOT = Path("/content/gdrive/MyDrive/CARDIUM")

ZIP_FILE = DRIVE_ROOT / "CARDIUM dataset-20260318T201738Z-3-001.zip"
TAR_FILE_1 = DRIVE_ROOT / "CARDIUM_dataset-002.tar.gz"
TAR_FILE_2 = DRIVE_ROOT / "cardium_images-003.tar.gz"

print("ZIP exists:", ZIP_FILE.exists(), ZIP_FILE)
print("TAR 1 exists:", TAR_FILE_1.exists(), TAR_FILE_1)
print("TAR 2 exists:", TAR_FILE_2.exists(), TAR_FILE_2)

ZIP exists: True /content/gdrive/MyDrive/CARDIUM/CARDIUM dataset-20260318T201738Z-3-001.zip
TAR 1 exists: True /content/gdrive/MyDrive/CARDIUM/CARDIUM_dataset-002.tar.gz
TAR 2 exists: True /content/gdrive/MyDrive/CARDIUM/cardium_images-003.tar.gz


In [ ]:
# ============================================================
# Working folders
# ============================================================

WORK_DIR = Path("/content/cardium_tabular_work")
ZIP_EXTRACT_DIR = WORK_DIR / "zip_extracted"
TAB_ENCODER_EXTRACT_DIR = WORK_DIR / "tabular_encoder_extracted"
EMBEDDING_OUT_DIR = DRIVE_ROOT / "CARDIUM_tabular_embeddings"

ZIP_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
TAB_ENCODER_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("WORK_DIR:", WORK_DIR)
print("ZIP_EXTRACT_DIR:", ZIP_EXTRACT_DIR)
print("TAB_ENCODER_EXTRACT_DIR:", TAB_ENCODER_EXTRACT_DIR)
print("EMBEDDING_OUT_DIR:", EMBEDDING_OUT_DIR)

WORK_DIR: /content/cardium_tabular_work
ZIP_EXTRACT_DIR: /content/cardium_tabular_work/zip_extracted
TAB_ENCODER_EXTRACT_DIR: /content/cardium_tabular_work/tabular_encoder_extracted
EMBEDDING_OUT_DIR: /content/gdrive/MyDrive/CARDIUM/CARDIUM_tabular_embeddings


In [ ]:
# ============================================================
# Extract only required tabular files from ZIP
# ============================================================

with zipfile.ZipFile(ZIP_FILE, "r") as z:
    zip_names = z.namelist()

    print("\nSearching required files...\n")

    for name in zip_names:
        if (
            "cardium_clinical_data_wnm_translated_final_cleaned.json" in name
            or "cardium_clinical_data_woe_wnm_standarized_f_normalized.json" in name
            or "tabular_encoder.tar.gz" in name
        ):
            print("Extracting:", name)
            z.extract(name, path=ZIP_EXTRACT_DIR)

print("\nDone extracting selected tabular files.")


Searching required files...

Extracting: CARDIUM dataset/cardium_clinical_data_wnm_translated_final_cleaned.json
Extracting: CARDIUM dataset/pesos_modelo/tabular_encoder.tar.gz
Extracting: CARDIUM dataset/cardium_clinical_data_woe_wnm_standarized_f_normalized.json

Done extracting selected tabular files.


In [ ]:
# ============================================================
# Verify extracted files
# ============================================================

for f in ZIP_EXTRACT_DIR.rglob("*"):
    print(f)

/content/cardium_tabular_work/zip_extracted/CARDIUM dataset
/content/cardium_tabular_work/zip_extracted/CARDIUM dataset/cardium_clinical_data_wnm_translated_final_cleaned.json
/content/cardium_tabular_work/zip_extracted/CARDIUM dataset/cardium_clinical_data_woe_wnm_standarized_f_normalized.json
/content/cardium_tabular_work/zip_extracted/CARDIUM dataset/pesos_modelo
/content/cardium_tabular_work/zip_extracted/CARDIUM dataset/pesos_modelo/tabular_encoder.tar.gz


In [ ]:
# ============================================================
# Define extracted file paths
# ============================================================

from pathlib import Path
import json

EXTRACTED_ROOT = Path("/content/cardium_tabular_work/zip_extracted/CARDIUM dataset")

RAW_TABULAR_JSON = EXTRACTED_ROOT / "cardium_clinical_data_wnm_translated_final_cleaned.json"
NORMALIZED_TABULAR_JSON = EXTRACTED_ROOT / "cardium_clinical_data_woe_wnm_standarized_f_normalized.json"
TABULAR_ENCODER_TAR = EXTRACTED_ROOT / "pesos_modelo" / "tabular_encoder.tar.gz"

print("RAW_TABULAR_JSON exists:", RAW_TABULAR_JSON.exists(), RAW_TABULAR_JSON)
print("NORMALIZED_TABULAR_JSON exists:", NORMALIZED_TABULAR_JSON.exists(), NORMALIZED_TABULAR_JSON)
print("TABULAR_ENCODER_TAR exists:", TABULAR_ENCODER_TAR.exists(), TABULAR_ENCODER_TAR)

RAW_TABULAR_JSON exists: True /content/cardium_tabular_work/zip_extracted/CARDIUM dataset/cardium_clinical_data_wnm_translated_final_cleaned.json
NORMALIZED_TABULAR_JSON exists: True /content/cardium_tabular_work/zip_extracted/CARDIUM dataset/cardium_clinical_data_woe_wnm_standarized_f_normalized.json
TABULAR_ENCODER_TAR exists: True /content/cardium_tabular_work/zip_extracted/CARDIUM dataset/pesos_modelo/tabular_encoder.tar.gz


In [ ]:
# ============================================================
# Preview normalized tabular JSON
# ============================================================

with open(NORMALIZED_TABULAR_JSON, "r") as f:
    normalized_data = json.load(f)

print("Number of patient records:", len(normalized_data))
print("\nFirst patient keys:")
print(normalized_data[0].keys())

Number of patient records: 1104

First patient keys:
dict_keys(['id', 'platelets', 'hemoglobin', 'hematocrit', 'white_blood_cells', 'neutrophils', 'age', 'gestational_week_of_imaging', 'body_mass_index', 'gestational_age', 'chromosomal_abnormality', 'screening_procedures', 'thromboembolic_risk', 'psychosocial_risk', 'depression_screening', 'tobacco_use', 'nutritional_deficiencies', 'alcohol_use', 'physical_activity', 'pregnancies', 'vaginal_births', 'cesarean_sections', 'miscarriages', 'ectopic_pregnancies', 'CHD', 'pathological_history', 'hereditary_history', 'pharmacological_history', 'ultrasound_date', 'pathological_history_woe', 'hereditary_history_woe', 'pharmacological_history_woe'])


In [ ]:
# ============================================================
# Inspect one normalized patient record
# ============================================================

sample_patient = normalized_data[0]

print("Patient ID:", sample_patient["id"])
print("CHD label:", sample_patient["CHD"])
print("platelets:", sample_patient["platelets"])
print("hemoglobin:", sample_patient["hemoglobin"])
print("age:", sample_patient["age"])
print("gestational_week_of_imaging:", sample_patient["gestational_week_of_imaging"])
print("body_mass_index:", sample_patient["body_mass_index"])
print("gestational_age:", sample_patient["gestational_age"])
print("pathological_history_woe:", sample_patient["pathological_history_woe"])
print("hereditary_history_woe:", sample_patient["hereditary_history_woe"])
print("pharmacological_history_woe:", sample_patient["pharmacological_history_woe"])

Patient ID: aedxf00593rrfgkl0
CHD label: 0
platelets: [1.722625372980866, -1]
hemoglobin: [2.0534567675331488, -1]
age: [0.8403826133191252, 0.8403826133191252, -1, -1, -1, -1, -1, -1, -1, -1]
gestational_week_of_imaging: [-1, -0.3393328163135717, -1, -1, -1, -1, -1, -1, -1, -1]
body_mass_index: [-1.2284176232937534, -0.3103091404706762, -1, -1, -1, -1, -1, -1, -1, -1]
gestational_age: [-1.5356057949686504, 0.07805777340672983, -1, -1, -1, -1, -1, -1, -1, -1]
pathological_history_woe: [0.535697528356774, 0.27091658046066064, 0.16946886153122423, 0.16946886153122423, 0.16946886153122423, 0.16946886153122423, 0.16946886153122423, 0.16946886153122423]
hereditary_history_woe: [0.03334123620432517, 0.58685449522787, 0.07999060278254538, 0.07999060278254538, 0.07999060278254538, 0.07999060278254538, 0.07999060278254538, 0.07999060278254538]
pharmacological_history_woe: [0.12699119158708194, 0.137116442595662, 0.137116442595662, 0.137116442595662, 0.137116442595662, 0.137116442595662, 0.13711

In [ ]:
# ============================================================
# Extract tabular encoder archive
# ============================================================

import tarfile

TAB_ENCODER_EXTRACT_DIR = Path("/content/cardium_tabular_work/tabular_encoder_extracted")
TAB_ENCODER_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with tarfile.open(TABULAR_ENCODER_TAR, "r:gz") as tar:
    names = tar.getnames()
    print("Files inside tabular_encoder.tar.gz:\n")
    for name in names[:100]:
        print(name)
    tar.extractall(path=TAB_ENCODER_EXTRACT_DIR)

print("\nTabular encoder extracted to:", TAB_ENCODER_EXTRACT_DIR)

Files inside tabular_encoder.tar.gz:

tabular_encoder
tabular_encoder/fold0_best_model.pth
tabular_encoder/fold2_best_model.pth
tabular_encoder/fold1_best_model.pth


/tmp/ipykernel_9311/1882591658.py:15: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=TAB_ENCODER_EXTRACT_DIR)



Tabular encoder extracted to: /content/cardium_tabular_work/tabular_encoder_extracted


In [ ]:
# ============================================================
# Find checkpoint files
# ============================================================

checkpoint_candidates = []
for ext in ("*.pth", "*.pt", "*.bin", "*.tar"):
    checkpoint_candidates.extend(TAB_ENCODER_EXTRACT_DIR.rglob(ext))

checkpoint_candidates = sorted(checkpoint_candidates)

print("Checkpoint candidates found:")
for p in checkpoint_candidates:
    print(p)

Checkpoint candidates found:
/content/cardium_tabular_work/tabular_encoder_extracted/tabular_encoder/fold0_best_model.pth
/content/cardium_tabular_work/tabular_encoder_extracted/tabular_encoder/fold1_best_model.pth
/content/cardium_tabular_work/tabular_encoder_extracted/tabular_encoder/fold2_best_model.pth


In [ ]:
# ============================================================
# ORIGINAL TABULAR MODEL
# from CARDIUM TabularTransformer.py
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import pandas as pd

class TabEncoder(nn.Module):
    """
    Original CARDIUM TabTransformer model.
    """
    def __init__(self, num_features, dim_embedding, num_heads, num_layers, dropout=0.3):
        super(TabEncoder, self).__init__()
        self.embedding = nn.Sequential(nn.Linear(1, dim_embedding))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim_embedding,
            nhead=num_heads,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.linear = nn.Sequential(
            nn.Linear(num_features * dim_embedding, dim_embedding),
            nn.LayerNorm(dim_embedding),
            nn.ReLU()
        )

        self.mlp = nn.Sequential(
            nn.Linear(dim_embedding, dim_embedding),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_embedding, dim_embedding // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_embedding // 2, dim_embedding // 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_embedding // 4, 1)
        )

    def forward(self, x):
        batch_size, num_features = x.shape
        x = x.unsqueeze(-1)
        x = self.embedding(x)
        x = self.transformer(x)
        x = x.view(batch_size, -1)
        x = self.linear(x)
        x = self.mlp(x)
        return x

In [ ]:
# ============================================================
# FEATURE ORDER
# compatibility wrapper for normalized JSON
# ============================================================

WOE_FEATURES = [
    "pathological_history_woe",
    "hereditary_history_woe",
    "pharmacological_history_woe",
]

FEATURE_ORDER = [
    "platelets",
    "hemoglobin",
    "hematocrit",
    "white_blood_cells",
    "neutrophils",
    "age",
    "gestational_week_of_imaging",
    "body_mass_index",
    "gestational_age",
    "chromosomal_abnormality",
    "screening_procedures",
    "thromboembolic_risk",
    "psychosocial_risk",
    "depression_screening",
    "tobacco_use",
    "nutritional_deficiencies",
    "alcohol_use",
    "physical_activity",
    "pregnancies",
    "vaginal_births",
    "cesarean_sections",
    "miscarriages",
    "ectopic_pregnancies",
]

def build_feature_vector(record):
    values = []

    for key in WOE_FEATURES:
        item = record.get(key, [])
        if isinstance(item, list):
            values.extend(item)

    for key in FEATURE_ORDER:
        item = record.get(key, -1)

        if isinstance(item, list):
            values.extend(item)
        elif isinstance(item, (int, float)):
            values.append(float(item))
        else:
            values.append(-1.0)

    return values

In [ ]:
# ============================================================
# BUILD PATIENT-LEVEL DATAFRAME
# ============================================================

rows = []

for patient in normalized_data:
    rows.append({
        "patient_id": patient["id"],
        "label": int(patient["CHD"]),
        "features": np.array(build_feature_vector(patient), dtype=np.float32)
    })

tabular_df = pd.DataFrame(rows)

print("tabular_df shape:", tabular_df.shape)
print("Example patient_id:", tabular_df.iloc[0]["patient_id"])
print("Example label:", tabular_df.iloc[0]["label"])
print("Feature vector length:", len(tabular_df.iloc[0]["features"]))

tabular_df shape: (1104, 3)
Example patient_id: aedxf00593rrfgkl0
Example label: 0
Feature vector length: 97


In [ ]:
# ============================================================
# ADDED: EMBEDDING EXTRACTOR
# this is the only new wrapper: stop at self.linear
# ============================================================

class TabularEmbeddingExtractor(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        batch_size, num_features = x.shape
        x = x.unsqueeze(-1)
        x = self.base_model.embedding(x)
        x = self.base_model.transformer(x)
        x = x.view(batch_size, -1)
        x = self.base_model.linear(x)
        return x

In [ ]:
# ============================================================
# LOAD CHECKPOINT ROBUSTLY
# ============================================================

def load_checkpoint_robust(model, checkpoint_path, device="cpu"):
    ckpt = torch.load(checkpoint_path, map_location=device)

    if isinstance(ckpt, dict):
        print("Checkpoint keys:", ckpt.keys())

        if "state_dict" in ckpt:
            state = ckpt["state_dict"]
        elif "model_state_dict" in ckpt:
            state = ckpt["model_state_dict"]
        elif "model" in ckpt and isinstance(ckpt["model"], dict):
            state = ckpt["model"]
        else:
            state = ckpt
    else:
        state = ckpt

    cleaned_state = {}
    for k, v in state.items():
        cleaned_state[k.replace("module.", "")] = v

    missing, unexpected = model.load_state_dict(cleaned_state, strict=False)
    print("Missing keys:", missing)
    print("Unexpected keys:", unexpected)

    model.eval()
    return model

In [ ]:
# ============================================================
# CHECKPOINTS
# ============================================================

CHECKPOINT_PATHS = checkpoint_candidates
print("Using checkpoints:")
for p in CHECKPOINT_PATHS:
    print(p)

Using checkpoints:
/content/cardium_tabular_work/tabular_encoder_extracted/tabular_encoder/fold0_best_model.pth
/content/cardium_tabular_work/tabular_encoder_extracted/tabular_encoder/fold1_best_model.pth
/content/cardium_tabular_work/tabular_encoder_extracted/tabular_encoder/fold2_best_model.pth


In [ ]:
# ============================================================
# MODEL SETTINGS
# these must match the checkpoint
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

num_features = len(tabular_df.iloc[0]["features"])
TAB_FEATURE_DIM = 128
TAB_NUM_HEADS = 8
TAB_NUM_LAYERS = 2      # FIXED: checkpoint has 2 layers, not 4
TAB_DROPOUT = 0.3

print("num_features:", num_features)

Using device: cuda
num_features: 97


In [ ]:
# ============================================================
# EXTRACT EMBEDDINGS FOR EACH FOLD CHECKPOINT
# save one parquet per fold
# ============================================================

X = np.stack(tabular_df["features"].values)
X = torch.tensor(X, dtype=torch.float32)

for ckpt_path in CHECKPOINT_PATHS:
    fold_name = ckpt_path.stem.replace("_best_model", "")
    print(f"\nProcessing checkpoint: {fold_name}")

    model = TabEncoder(
        num_features=num_features,
        dim_embedding=TAB_FEATURE_DIM,
        num_heads=TAB_NUM_HEADS,
        num_layers=TAB_NUM_LAYERS,
        dropout=TAB_DROPOUT
    ).to(device)

    model = load_checkpoint_robust(model, ckpt_path, device=device)
    extractor = TabularEmbeddingExtractor(model).to(device)

    all_embeddings = []

    extractor.eval()
    with torch.no_grad():
        batch_size = 64
        for i in range(0, len(X), batch_size):
            batch = X[i:i + batch_size].to(device)
            emb = extractor(batch).cpu().numpy()
            all_embeddings.append(emb)

    tabular_embeddings = np.vstack(all_embeddings)
    print("Embedding matrix shape:", tabular_embeddings.shape)

    emb_cols = [f"emb_{i}" for i in range(tabular_embeddings.shape[1])]
    emb_df = pd.DataFrame(tabular_embeddings, columns=emb_cols)

    final_tabular_df = pd.concat(
        [
            tabular_df[["patient_id", "label"]].reset_index(drop=True),
            emb_df.reset_index(drop=True)
        ],
        axis=1
    )

    out_file = EMBEDDING_OUT_DIR / f"cardium_tabular_embeddings_{fold_name}.parquet"
    final_tabular_df.to_parquet(out_file, index=False)

    print("Saved:", out_file)
    print("Final shape:", final_tabular_df.shape)


Processing checkpoint: fold0
Checkpoint keys: odict_keys(['embedding.0.weight', 'embedding.0.bias', 'transformer.layers.0.self_attn.in_proj_weight', 'transformer.layers.0.self_attn.in_proj_bias', 'transformer.layers.0.self_attn.out_proj.weight', 'transformer.layers.0.self_attn.out_proj.bias', 'transformer.layers.0.linear1.weight', 'transformer.layers.0.linear1.bias', 'transformer.layers.0.linear2.weight', 'transformer.layers.0.linear2.bias', 'transformer.layers.0.norm1.weight', 'transformer.layers.0.norm1.bias', 'transformer.layers.0.norm2.weight', 'transformer.layers.0.norm2.bias', 'transformer.layers.1.self_attn.in_proj_weight', 'transformer.layers.1.self_attn.in_proj_bias', 'transformer.layers.1.self_attn.out_proj.weight', 'transformer.layers.1.self_attn.out_proj.bias', 'transformer.layers.1.linear1.weight', 'transformer.layers.1.linear1.bias', 'transformer.layers.1.linear2.weight', 'transformer.layers.1.linear2.bias', 'transformer.layers.1.norm1.weight', 'transformer.layers.1.norm

In [ ]:
# ============================================================
# OPTIONAL: AVERAGE EMBEDDINGS ACROSS THE 3 FOLDS
# gives one final embedding per patient
# ============================================================

parquet_files = sorted(EMBEDDING_OUT_DIR.glob("cardium_tabular_embeddings_fold*.parquet"))
print("Parquet files found:")
for p in parquet_files:
    print(p)

dfs = [pd.read_parquet(p) for p in parquet_files]

base = dfs[0][["patient_id", "label"]].copy()

emb_arrays = []
for df in dfs:
    emb_cols = [c for c in df.columns if c.startswith("emb_")]
    emb_arrays.append(df[emb_cols].values)

mean_emb = np.mean(np.stack(emb_arrays, axis=0), axis=0)

mean_cols = [f"emb_{i}" for i in range(mean_emb.shape[1])]
mean_df = pd.concat(
    [
        base.reset_index(drop=True),
        pd.DataFrame(mean_emb, columns=mean_cols)
    ],
    axis=1
)

mean_out = EMBEDDING_OUT_DIR / "cardium_tabular_embeddings_mean_of_3_folds.parquet"
mean_df.to_parquet(mean_out, index=False)

print("Saved averaged embeddings:", mean_out)
print("Shape:", mean_df.shape)

Parquet files found:
/content/gdrive/MyDrive/CARDIUM/CARDIUM_tabular_embeddings/cardium_tabular_embeddings_fold0.parquet
/content/gdrive/MyDrive/CARDIUM/CARDIUM_tabular_embeddings/cardium_tabular_embeddings_fold1.parquet
/content/gdrive/MyDrive/CARDIUM/CARDIUM_tabular_embeddings/cardium_tabular_embeddings_fold2.parquet
Saved averaged embeddings: /content/gdrive/MyDrive/CARDIUM/CARDIUM_tabular_embeddings/cardium_tabular_embeddings_mean_of_3_folds.parquet
Shape: (1104, 130)
